# 03 — Imbalance Strategies

**This notebook is what sets a senior DS apart** — methodically compare strategies for extreme class imbalance.

**Fraud rate**: ~0.57% → 1:175 imbalance ratio

### Strategies Compared
| Strategy | Description | When to Use |
|----------|-------------|-------------|
| **No resampling** | Raw imbalanced data | With `class_weight='balanced'` |
| **class_weight** | Algorithm-level weighting | Fast, no data modification |
| **RandomUnderSampler** | Remove majority class | Large datasets where speed matters |
| **SMOTE** | Synthetic minority oversampling | Small datasets |
| **SMOTEENN** | SMOTE + Edited Nearest Neighbors | Noisy datasets |

### Key Insight
> For fraud detection: **PR-AUC and Recall** matter more than accuracy or F1.
> A model that misses 50% of fraud is far worse than one that has 30% false alarm rate.

---

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    average_precision_score, f1_score, fbeta_score,
    recall_score, precision_score, precision_recall_curve
)
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (14, 5), 'figure.dpi': 110,
                     'axes.spines.top': False, 'axes.spines.right': False})

from src.data.loader import load_train_test
from src.data.features import engineer_features, split_features_target, NUMERIC_FEATURES, CATEGORICAL_FEATURES

df_train_raw, df_test_raw = load_train_test(
    '../data/fraudTrain.csv', '../data/fraudTest.csv',
    train_sample_size=50_000,  # Smaller for faster imbalance experiments
    smoke_test_n=10_000,
)
df_train = engineer_features(df_train_raw)
df_test  = engineer_features(df_test_raw)
X_train, y_train = split_features_target(df_train)
X_test,  y_test  = split_features_target(df_test)

print(f'Train fraud: {y_train.sum():,} ({y_train.mean():.4%})')
print(f'Test  fraud: {y_test.sum():,}  ({y_test.mean():.4%})')

## Step 1: Visualize the Imbalance Problem

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class counts
axes[0].bar(['Legitimate', 'Fraud'],
            [y_train.value_counts()[0], y_train.value_counts()[1]],
            color=['#2196F3', '#F44336'], alpha=0.85)
axes[0].set_yscale('log'); axes[0].set_ylabel('Count (log scale)')
axes[0].set_title(f'Class Distribution (Imbalance ≈ 1:{int(1/y_train.mean())})', fontweight='bold')
for i, v in enumerate([y_train.value_counts()[0], y_train.value_counts()[1]]):
    axes[0].text(i, v*1.5, f'{v:,}', ha='center', fontweight='bold')

# What naive predictions look like
naive_accuracy = 1 - y_train.mean()
naive_recall = 0.0
actual_recall_target = 0.9  # We want to catch 90% of fraud
axes[1].barh(['Naive (Predict All Legit)', 'Target (90% Fraud Recall)'],
             [naive_accuracy*100, actual_recall_target*100],
             color=['grey', '#4CAF50'], alpha=0.85)
axes[1].set_xlabel('Metric Value (%)')
axes[1].axvline(naive_accuracy*100, color='red', ls='--', lw=1.5, label=f'Naive accuracy={naive_accuracy:.2%}')
axes[1].set_title('Accuracy is Misleading for Fraud!', fontweight='bold')
axes[1].legend(fontsize=9)
for i, v in enumerate([naive_accuracy*100, actual_recall_target*100]):
    axes[1].text(v+1, i, f'{v:.1f}%', va='center', fontsize=11, fontweight='bold')

plt.suptitle('The Class Imbalance Problem', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'Naive classifier accuracy: {naive_accuracy:.4%}')
print('This is useless — it catches ZERO fraud!')

## Step 2: Build Evaluation Pipeline

In [ ]:
def build_lr_pipeline(class_weight='balanced'):
    """Simple LR pipeline for fast imbalance strategy comparison."""
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
    ])
    return Pipeline([('preprocessor', preprocessor),
                     ('classifier', LogisticRegression(class_weight=class_weight,
                                                        max_iter=500, C=0.1, solver='lbfgs'))])

def evaluate_strategy(pipeline, X_tr, y_tr, X_te, y_te, name):
    pipeline.fit(X_tr, y_tr)
    y_prob = pipeline.predict_proba(X_te)[:, 1]
    # Optimize threshold for F2
    best_f2, best_t = 0, 0.5
    for t in np.linspace(0.01, 0.99, 100):
        f2 = fbeta_score(y_te, (y_prob >= t).astype(int), beta=2, zero_division=0)
        if f2 > best_f2: best_f2, best_t = f2, t
    y_pred = (y_prob >= best_t).astype(int)
    return {
        'Strategy': name,
        'PR-AUC': round(average_precision_score(y_te, y_prob), 4),
        'F2-Score': round(best_f2, 4),
        'Recall': round(recall_score(y_te, y_pred, zero_division=0), 4),
        'Precision': round(precision_score(y_te, y_pred, zero_division=0), 4),
        'F1-Score': round(f1_score(y_te, y_pred, zero_division=0), 4),
        'Threshold': round(best_t, 3),
        '_probs': y_prob,
    }

results = {}
print('Evaluating strategies...')

## Strategy 1: No Resampling (Default Threshold = 0.5)

In [ ]:
res = evaluate_strategy(build_lr_pipeline(class_weight=None),
                         X_train, y_train, X_test, y_test, 'No Resampling (LR)')
results['no_resampling'] = res
print(f"PR-AUC: {res['PR-AUC']:.4f} | F2: {res['F2-Score']:.4f} | Recall: {res['Recall']:.4f}")

## Strategy 2: class_weight='balanced'

In [ ]:
res = evaluate_strategy(build_lr_pipeline(class_weight='balanced'),
                         X_train, y_train, X_test, y_test, "class_weight='balanced'")
results['class_weight'] = res
print(f"PR-AUC: {res['PR-AUC']:.4f} | F2: {res['F2-Score']:.4f} | Recall: {res['Recall']:.4f}")

## Strategy 3: Random Undersampling

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Apply resampling on the raw feature columns
preprocessor_fit = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer()), ('sc', StandardScaler())]), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
])
X_tr_enc = preprocessor_fit.fit_transform(X_train)
X_te_enc = preprocessor_fit.transform(X_test)

rus = RandomUnderSampler(random_state=42)
X_res, y_res = rus.fit_resample(X_tr_enc, y_train)
print(f'After undersampling: {len(X_res):,} samples | fraud={y_res.mean():.2%}')

from sklearn.linear_model import LogisticRegression as LR
clf = LR(max_iter=500, C=0.1, solver='lbfgs')
clf.fit(X_res, y_res)
y_prob = clf.predict_proba(X_te_enc)[:, 1]

best_f2, best_t = 0, 0.5
for t in np.linspace(0.01, 0.99, 100):
    f2 = fbeta_score(y_test, (y_prob >= t).astype(int), beta=2, zero_division=0)
    if f2 > best_f2: best_f2, best_t = f2, t
y_pred = (y_prob >= best_t).astype(int)
res_rus = {'Strategy': 'RandomUnderSampler', 'PR-AUC': round(average_precision_score(y_test, y_prob),4),
           'F2-Score': round(best_f2,4), 'Recall': round(recall_score(y_test,y_pred,zero_division=0),4),
           'Precision': round(precision_score(y_test,y_pred,zero_division=0),4),
           'F1-Score': round(f1_score(y_test,y_pred,zero_division=0),4),
           'Threshold': round(best_t,3), '_probs': y_prob}
results['undersampling'] = res_rus
print(f"PR-AUC: {res_rus['PR-AUC']:.4f} | F2: {res_rus['F2-Score']:.4f} | Recall: {res_rus['Recall']:.4f}")

## Strategy 4: SMOTE Oversampling

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(k_neighbors=5, random_state=42)
X_smote, y_smote = smote.fit_resample(X_tr_enc, y_train)
print(f'After SMOTE: {len(X_smote):,} samples | fraud={y_smote.mean():.2%}')

clf2 = LR(max_iter=500, C=0.1, solver='lbfgs')
clf2.fit(X_smote, y_smote)
y_prob2 = clf2.predict_proba(X_te_enc)[:, 1]

best_f2, best_t = 0, 0.5
for t in np.linspace(0.01, 0.99, 100):
    f2 = fbeta_score(y_test, (y_prob2 >= t).astype(int), beta=2, zero_division=0)
    if f2 > best_f2: best_f2, best_t = f2, t
y_pred2 = (y_prob2 >= best_t).astype(int)
res_smote = {'Strategy': 'SMOTE', 'PR-AUC': round(average_precision_score(y_test, y_prob2),4),
             'F2-Score': round(best_f2,4), 'Recall': round(recall_score(y_test,y_pred2,zero_division=0),4),
             'Precision': round(precision_score(y_test,y_pred2,zero_division=0),4),
             'F1-Score': round(f1_score(y_test,y_pred2,zero_division=0),4),
             'Threshold': round(best_t,3), '_probs': y_prob2}
results['smote'] = res_smote
print(f"PR-AUC: {res_smote['PR-AUC']:.4f} | F2: {res_smote['F2-Score']:.4f} | Recall: {res_smote['Recall']:.4f}")

## Step 3: Comparison

In [ ]:
comparison_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != '_probs'}
    for r in results.values()
]).sort_values('PR-AUC', ascending=False)

print('=== IMBALANCE STRATEGY COMPARISON ===')
print(comparison_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
COLORS = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

for ax, metric in zip(axes, ['PR-AUC', 'F2-Score', 'Recall']):
    vals = comparison_df[metric].tolist()
    names = comparison_df['Strategy'].tolist()
    bars = ax.bar(names, vals, color=COLORS[:len(vals)], alpha=0.85)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Imbalance Strategy Comparison (Logistic Regression baseline)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# PR curves for all strategies
fig, ax = plt.subplots(figsize=(10, 7))
baseline_rate = y_test.mean()
ax.axhline(baseline_rate, color='grey', ls='--', lw=1.5, label=f'Baseline={baseline_rate:.4%}')

for i, (key, res) in enumerate(results.items()):
    prec, rec, _ = precision_recall_curve(y_test, res['_probs'])
    ap = average_precision_score(y_test, res['_probs'])
    ax.plot(rec, prec, lw=2, color=COLORS[i], label=f"{res['Strategy']} (AP={ap:.4f})")

ax.set_xlabel('Recall (Fraud Caught)'); ax.set_ylabel('Precision (Alert Accuracy)')
ax.set_title('Precision-Recall Curves: Strategy Comparison', fontsize=12, fontweight='bold')
ax.legend(fontsize=10); plt.tight_layout(); plt.show()

## Conclusions

| Strategy | Verdict | Recommendation |
|----------|---------|----------------|
| No resampling | ❌ | Poor recall without class weight |
| `class_weight='balanced'` | ✅ | Best baseline — fast, no data leakage risk |
| RandomUnderSampler | ⚠️ | Loses data; OK for very large datasets |
| SMOTE | ✅ | Good for moderate datasets; risk of overfitting to synthetic points |

**Decision**: Use `class_weight='balanced'` for gradient boosting models, combined with F2-score threshold optimization. For XGBoost, use `scale_pos_weight` (automatic ratio). For LightGBM, use `is_unbalance=True`.

> **Next**: → `04_Model_Experiments.ipynb`